# Linear Regression

In [66]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [3]:
df=pd.read_csv('../data/kc_house_data.csv')

In [4]:
df.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [5]:
df.shape

(21613, 21)

In [9]:
# df.drop('date',axis=1,inplace=True)
df.drop('id',axis=1,inplace=True)

In [7]:
df['waterfront'].value_counts()
df['view'].value_counts()

view
0    19489
2      963
3      510
1      332
4      319
Name: count, dtype: int64

In [6]:
df['age']=df['yr_built'].max()-df['yr_built']

In [10]:
df.drop('yr_built',axis=1,inplace=True)

In [11]:
x=df['yr_renovated'].values != 0
xd=pd.DataFrame(x)
xd.value_counts()

0    
False    20699
True       914
Name: count, dtype: int64

In [12]:
df.drop('yr_renovated',axis=1,inplace=True)

In [13]:
df.head()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,zipcode,lat,long,sqft_living15,sqft_lot15,age
0,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,98178,47.5112,-122.257,1340,5650,60
1,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,98125,47.7210,-122.319,1690,7639,64
2,180000.0,2,1.00,770,10000,1.0,0,0,3,6,770,0,98028,47.7379,-122.233,2720,8062,82
3,604000.0,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,98136,47.5208,-122.393,1360,5000,50
4,510000.0,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,98074,47.6168,-122.045,1800,7503,28


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   price          21613 non-null  float64
 1   bedrooms       21613 non-null  int64  
 2   bathrooms      21613 non-null  float64
 3   sqft_living    21613 non-null  int64  
 4   sqft_lot       21613 non-null  int64  
 5   floors         21613 non-null  float64
 6   waterfront     21613 non-null  int64  
 7   view           21613 non-null  int64  
 8   condition      21613 non-null  int64  
 9   grade          21613 non-null  int64  
 10  sqft_above     21613 non-null  int64  
 11  sqft_basement  21613 non-null  int64  
 12  zipcode        21613 non-null  int64  
 13  lat            21613 non-null  float64
 14  long           21613 non-null  float64
 15  sqft_living15  21613 non-null  int64  
 16  sqft_lot15     21613 non-null  int64  
 17  age            21613 non-null  int64  
dtypes: flo

In [16]:
x=df.drop('price',axis=1,inplace=False)
y=df['price']

In [17]:
y.head()

0    221900.0
1    538000.0
2    180000.0
3    604000.0
4    510000.0
Name: price, dtype: float64

In [18]:
df.describe()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,zipcode,lat,long,sqft_living15,sqft_lot15,age
count,2.161300e+04,21613.000000,21613.000000,21613.000000,2.161300e+04,21613.000000,21613.000000,21613.000000,21613.000000,21613.000000,21613.000000,21613.000000,21613.000000,21613.000000,21613.000000,21613.000000,21613.000000,21613.000000
mean,5.400881e+05,3.370842,2.114757,2079.899736,1.510697e+04,1.494309,0.007542,0.234303,3.409430,7.656873,1788.390691,291.509045,98077.939805,47.560053,-122.213896,1986.552492,12768.455652,43.994864
std,3.671272e+05,0.930062,0.770163,918.440897,4.142051e+04,0.539989,0.086517,0.766318,0.650743,1.175459,828.090978,442.575043,53.505026,0.138564,0.140828,685.391304,27304.179631,29.373411
min,7.500000e+04,0.000000,0.000000,290.000000,5.200000e+02,1.000000,0.000000,0.000000,1.000000,1.000000,290.000000,0.000000,98001.000000,47.155900,-122.519000,399.000000,651.000000,0.000000
25%,3.219500e+05,3.000000,1.750000,1427.000000,5.040000e+03,1.000000,0.000000,0.000000,3.000000,7.000000,1190.000000,0.000000,98033.000000,47.471000,-122.328000,1490.000000,5100.000000,18.000000
50%,4.500000e+05,3.000000,2.250000,1910.000000,7.618000e+03,1.500000,0.000000,0.000000,3.000000,7.000000,1560.000000,0.000000,98065.000000,47.571800,-122.230000,1840.000000,7620.000000,40.000000
75%,6.450000e+05,4.000000,2.500000,2550.000000,1.068800e+04,2.000000,0.000000,0.000000,4.000000,8.000000,2210.000000,560.000000,98118.000000,47.678000,-122.125000,2360.000000,10083.000000,64.000000
max,7.700000e+06,33.000000,8.000000,13540.000000,1.651359e+06,3.500000,1.000000,4.000000,5.000000,13.000000,9410.000000,4820.000000,98199.000000,47.777600,-121.315000,6210.000000,871200.000000,115.000000


In [19]:
x.head()

,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,zipcode,lat,long,sqft_living15,sqft_lot15,age
0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,98178,47.5112,-122.257,1340,5650,60
1,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,98125,47.7210,-122.319,1690,7639,64
2,2,1.00,770,10000,1.0,0,0,3,6,770,0,98028,47.7379,-122.233,2720,8062,82
3,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,98136,47.5208,-122.393,1360,5000,50
4,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,98074,47.6168,-122.045,1800,7503,28


In [49]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [21]:
df.shape

(21613, 18)

In [22]:
x_train.shape

(17290, 17)

In [50]:
x_test.shape

(4323, 17)

In [51]:
y_train.shape

(17290,)

In [52]:
y_test.shape

(4323,)

In [23]:
model=LinearRegression()
model.fit(x_train,y_train)

LinearRegression()

In [24]:
y_pr=pd.DataFrame(model.predict(x_test))

In [25]:
type(y_pr)

pandas.core.frame.DataFrame

## evaluation of the model

In [29]:
mae=mean_absolute_error(y_true=y_test,y_pred=y_pr)
print(mae)

127649.68509875197


In [36]:
mse=mean_squared_error(y_test,y_pr)
mse

45219103217.24472

In [37]:
rmse=np.sqrt(mse)
rmse

np.float64(212647.83849652627)

In [30]:
r2=r2_score(y_true=y_test,y_pred=y_pr)
r2

0.7008857876177697

In [31]:
df.shape

(21613, 18)

### adjusted r2

In [35]:
ad_r=1-((1-r2)*(21613-1)/(21613-1-18))
ad_r

0.7006364565154782

## conclusion
From above as we can see that the range of the errors shown is very large, This is because the values are not scaled and are in the greater range.

as the data is multi dimensinal and also there are issues seen in the evaluation metrics of the model. I can take the following steps to increase the performance of the model  
- apply pca to reduce the dimensionality.  
- scale the features to a small range.  

for this purpose i will create a new dataframe and then apply all the techniques on that dataframe so that will get the good results at the end.  

In [67]:
scaler=StandardScaler()
scaler.fit(x_train)
x_train_scaled=scaler.transform(x_train)
x_test_scaled=scaler.transform(x_test)

In [87]:
pca=PCA(n_components=12)
pca.fit(x_train_scaled)

PCA(n_components=12)

In [84]:
pca.components_

array([[ 0.24520562,  0.36459949,  0.39491911,  0.10681127,  0.23691383,
         0.04113728,  0.10710002, -0.08751624,  0.36761044,  0.39386729,
         0.07908693, -0.16334883, -0.00281539,  0.19113143,  0.35544324,
         0.1087761 , -0.25436976],
       [ 0.17726666,  0.09793606,  0.20091502, -0.179266  , -0.13023449,
         0.19263711,  0.3371739 ,  0.21643104,  0.09913351, -0.01953915,
         0.45208894,  0.33547544,  0.24237944, -0.386158  ,  0.08415846,
        -0.18554003,  0.30882398],
       [ 0.00388135, -0.10733992,  0.05259913,  0.5371985 , -0.34897055,
         0.12175856,  0.16720774,  0.25982568, -0.08633843, -0.05245822,
         0.20701738, -0.14839084, -0.14446173,  0.15445467,  0.06012039,
         0.54160833,  0.21972499],
       [-0.33705361, -0.05077787, -0.08238221,  0.27625944,  0.29076775,
         0.34936199,  0.29381428, -0.34943401,  0.0907488 ,  0.04907774,
        -0.2622787 ,  0.3806391 ,  0.21028652, -0.20008398, -0.03688608,
         0.27146186

In [85]:
pca.explained_variance_

array([5.20596149, 2.16526522, 1.85631681, 1.3506692 , 1.21201079,
       0.87221908, 0.83227398, 0.66980328, 0.63247946, 0.49853814])

In [88]:
np.cumsum(pca.explained_variance_ratio_)

array([0.30621532, 0.43357649, 0.54276528, 0.62221182, 0.69350245,
       0.74480648, 0.79376095, 0.83315886, 0.87036138, 0.89968546,
       0.92369436, 0.94413775])

In [89]:
x_train_pca=pca.transform(pd.DataFrame(x_train_scaled))
x_test_pca=pca.transform(pd.DataFrame(x_test_scaled))

In [90]:
lr=LinearRegression()
lr.fit(x_train_pca,y_train)

LinearRegression()

In [96]:
y_train.shape

(17290,)

In [97]:
y_pr_pca.shape

(4323,)

In [93]:
y_pr_pca=lr.predict(x_test_pca)
r2_pca=r2_score(y_pred=y_pr_pca,y_true=y_test)
print(r2_pca)

0.6906568217058418


In [98]:
mae2=mean_absolute_error(y_pred=y_pr_pca,y_true=y_test)
mae2

130155.62675094913

even after the data is dimensionally reduced and even after that the model performance is not improved then the issue can be with the data itself.  


In [99]:
df.corr()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,zipcode,lat,long,sqft_living15,sqft_lot15,age
price,1.000000,0.308350,0.525138,0.702035,0.089661,0.256794,0.266369,0.397293,0.036362,0.667434,0.605567,0.323816,-0.053203,0.307003,0.021626,0.585379,0.082447,-0.054012
bedrooms,0.308350,1.000000,0.515884,0.576671,0.031703,0.175429,-0.006582,0.079532,0.028472,0.356967,0.477600,0.303093,-0.152668,-0.008931,0.129473,0.391638,0.029244,-0.154178
bathrooms,0.525138,0.515884,1.000000,0.754665,0.087740,0.500653,0.063744,0.187737,-0.124982,0.664983,0.685342,0.283770,-0.203866,0.024573,0.223042,0.568634,0.087175,-0.506019
sqft_living,0.702035,0.576671,0.754665,1.000000,0.172826,0.353949,0.103818,0.284611,-0.058753,0.762704,0.876597,0.435043,-0.199430,0.052529,0.240223,0.756420,0.183286,-0.318049
sqft_lot,0.089661,0.031703,0.087740,0.172826,1.000000,-0.005201,0.021604,0.074710,-0.008958,0.113621,0.183512,0.015286,-0.129574,-0.085683,0.229521,0.144608,0.718557,-0.053080
floors,0.256794,0.175429,0.500653,0.353949,-0.005201,1.000000,0.023698,0.029444,-0.263768,0.458183,0.523885,-0.245705,-0.059121,0.049614,0.125419,0.279885,-0.011269,-0.489319
waterfront,0.266369,-0.006582,0.063744,0.103818,0.021604,0.023698,1.000000,0.401857,0.016653,0.082775,0.072075,0.080588,0.030285,-0.014274,-0.041910,0.086463,0.030703,0.026161
view,0.397293,0.079532,0.187737,0.284611,0.074710,0.029444,0.401857,1.000000,0.045990,0.251321,0.167649,0.276947,0.084827,0.006157,-0.078400,0.280439,0.072575,0.053440
condition,0.036362,0.028472,-0.124982,-0.058753,-0.008958,-0.263768,0.016653,0.045990,1.000000,-0.144674,-0.158214,0.174105,0.003026,-0.014941,-0.106500,-0.092824,-0.003406,0.361417
grade,0.667434,0.356967,0.664983,0.762704,0.113621,0.458183,0.082775,0.251321,-0.144674,1.000000,0.755923,0.168392,-0.184862,0.114084,0.198372,0.713202,0.119248,-0.446963
